# Averis X Monash Hackathon
**Team dareDEVils**

1. Classification Pipeline testing
2. Comparison & Analytics Strategy testing

Install and Import all the required Libraries and Modules

In [3]:
! pip install pandas
! pip install spacy

   ---------------------------------------- 0.0/15.2 MB ? eta -:--:--
   ----------- ---------------------------- 4.5/15.2 MB 30.0 MB/s eta 0:00:01
   -------------------------- ------------- 10.0/15.2 MB 25.9 MB/s eta 0:00:01
   ---------------------------------------- 15.2/15.2 MB 29.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/652.5 kB ? eta -:--:--
   --------------------------------------- 652.5/652.5 kB 24.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 47.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.2 MB ? eta -:--:--
   ---------------------------------------- 6.2/6.2 MB 31.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Uninstalling click-8.1.7:
      Successfully uninstalled click-8.1.7


In [4]:
# for data loading
import pandas as pd
import os
import glob
import json

# for classification
import spacy

The data is to be retrieved either directly from data_v2 folder or using a uvicorn FastAPI server with endpoints:

**The endpoints**
 
| Method | Path | Returns |
|---|---|---|
| GET | `/health` | `{"status","emails","scoring_available"}` |
| GET | `/emails` | list of all 520 email records |
| GET | `/emails/{email_id}` | one record, e.g. `/emails/email_004` |
| GET | `/attachments/{path}` | the raw file bytes |
| GET | `/sample_submission` | the exact output shape, all 520 keys |
| POST | `/submit` | scoreboard JSON |
| GET | `/ground_truth` | 404 unless `REVEAL_GT=1` — judges only |
 
`{path}` is the attachment string **minus** the `attachments/` prefix already in
the URL, so `attachments/email_004_SI.txt` → `GET /attachments/email_004_SI.txt`.
Just concatenate: `base_url + "/" + att_string` gives the right URL either way.

The expected input is an email for the given dataset which haas 5 fields and is a json of the format:
```bash
{
  "email_id": "email_XXX",
  "from": "abc1234@pqrs.xxx",
  "subject": "XXXX YYYY ZZZZ",
  "body": "lorem ipsum ........",
  "attachments": ["attachments/email_XXX_SI.yyy", "attachments/email_XXX_BL.yyy"]
}
```

Load the input data directly from data_v2/inbox using pandas

In [7]:
# List out all the filenames of the email json files in inbox directory
inbox_dir = "data_v2/inbox"
mail_files = os.path.join(inbox_dir, "*.json") # all mail as json files
mail_files_list = glob.glob(mail_files) # list of all json files

# for each mail file, read and add the mail information to dataframe
mail_data = []
for mail_file in mail_files_list:
    with open(mail_file, "r") as mfile:
        mail_data.append(pd.json_normalize(json.loads(mfile.read())))

# convert the extracted json fields data to dataframe
mail_df = pd.concat(mail_data)

## Classification

**Current Plan:**

1. Spacy similarity matching (Primary Comparison)
2. LLM Call (Confidence Score based Fallback)

In [8]:
"""
Classification Pipeline
"""

# mail has to be classified into one of these categories
# {label: explanation}
mail_categories = {
    "BL_COMPARISON": "Comparison requested for the Bill of Landing (BL) and Shipping Instruction (SI)",
    "SI_REQUEST": "Request for new Shipping Information (SI)",
    "INVOICE_QUERY": "Query about Invoice",
    "GENERAL": "General Messages",
    "SPAM": "Spam mails (unwanted messages)"
}

# Comparison

**Current Plan:**
Multi-Stage analysis and escalation as required
1. REGEX + Levenshtein distance
2. Vector Embeddings for grouping
3. Further Extraction and Numerical Units, Product Features and Dimensional Comparison
4. LLM Call (Confidence Score based Fallback)
5. Human in the Loop

In [ ]:
"""
Comparison
"""

''